# Lab 2 — Oxford Pet Semantic Segmentation
U-Net / ResNet34-UNet 訓練與推論

## 使用前準備
1. 執行階段 → 變更執行階段類型 → **GPU**
2. Google Drive 的 `MyDrive/lab2/` 裡要有：
   ```
   MyDrive/lab2/
   ├── dataset/oxford-iiit-pet/       ← 圖片和 trimaps（只需上傳一次）
   ├── nycu-2026-spring-dl-lab2-unet/ ← split txt 檔
   └── binary-semantic-segmentation-res-net-34-u-net/ ← split txt 檔
   ```
3. 程式碼從 GitHub clone，**不需要上傳 src/**

In [ ]:
# ── 1. 確認 GPU ──────────────────────────────────────────────────────
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 2. Clone repo（只拿程式碼，很快）────────────────────────────────
import os

if not os.path.exists('/content/NYCU-Deep-Learing'):
    !git clone https://github.com/InnisChen/NYCU-Deep-Learing /content/NYCU-Deep-Learing
else:
    print('Repo already cloned, pulling latest...')
    !git -C /content/NYCU-Deep-Learing pull

%cd /content/NYCU-Deep-Learing/lab2
import sys
sys.path.insert(0, 'src')
print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── 3. 掛載 Google Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── 4. Symlink dataset 和 split 資料夾（不複製，直接指向 Drive）──────
DRIVE_LAB2 = '/content/drive/MyDrive/lab2'

links = {
    f'{DRIVE_LAB2}/dataset':                                          'dataset',
    f'{DRIVE_LAB2}/nycu-2026-spring-dl-lab2-unet':                   'nycu-2026-spring-dl-lab2-unet',
    f'{DRIVE_LAB2}/binary-semantic-segmentation-res-net-34-u-net':   'binary-semantic-segmentation-res-net-34-u-net',
}

for src, dst in links.items():
    if os.path.exists(dst) or os.path.islink(dst):
        print(f'  already exists: {dst}')
    elif os.path.exists(src):
        os.symlink(src, dst)
        print(f'  linked: {dst} → {src}')
    else:
        print(f'  WARNING: source not found: {src}')

In [ ]:
# ── 5. 確認資料集結構 ─────────────────────────────────────────────────
dataset_root = 'dataset/oxford-iiit-pet'
print('Images :', len(os.listdir(f'{dataset_root}/images')), 'files')
print('Trimaps:', len(os.listdir(f'{dataset_root}/annotations/trimaps')), 'files')

for split_dir in ['nycu-2026-spring-dl-lab2-unet', 'binary-semantic-segmentation-res-net-34-u-net']:
    if os.path.exists(split_dir):
        print(f'{split_dir}/: {os.listdir(split_dir)}')
    else:
        print(f'MISSING: {split_dir}/')

## 6. 更新程式碼
本機改完程式碼 → `git push` → 執行下面這個 cell

In [ ]:
# ── 6. 拉最新程式碼 ───────────────────────────────────────────────────
!git pull
!git log --oneline -3

## 7. 訓練 — 競賽一：UNet
Kaggle 競賽：`nycu-2026-spring-dl-lab2-unet`

In [ ]:
# ── 7. UNet 訓練參數 ──────────────────────────────────────────────────
import os
EPOCHS     = 50
BATCH_SIZE = 16
LR         = 1e-3
DATA_PATH  = 'dataset/oxford-iiit-pet'
SAVE_PATH  = '/content/drive/MyDrive/lab2/saved_models'

cmd_unet = (f'python src/train.py'
            f' --model unet'
            f' --split_dir nycu-2026-spring-dl-lab2-unet'
            f' --data_path {DATA_PATH}'
            f' --save_path {SAVE_PATH}'
            f' --epochs {EPOCHS}'
            f' --batch_size {BATCH_SIZE}'
            f' --learning_rate {LR}')
print('Command:', cmd_unet)

In [ ]:
# ── 執行 UNet 訓練 ────────────────────────────────────────────────────
os.makedirs(SAVE_PATH, exist_ok=True)
!{cmd_unet}

## 8. 訓練 — 競賽二：ResNet34-UNet
Kaggle 競賽：`binary-semantic-segmentation-res-net-34-u-net`

In [ ]:
# ── 8. ResNet34-UNet 訓練參數 ─────────────────────────────────────────
import os
EPOCHS     = 50
BATCH_SIZE = 16
LR         = 1e-3
DATA_PATH  = 'dataset/oxford-iiit-pet'
SAVE_PATH  = '/content/drive/MyDrive/lab2/saved_models'

cmd_res = (f'python src/train.py'
           f' --model resnet34_unet'
           f' --split_dir binary-semantic-segmentation-res-net-34-u-net'
           f' --data_path {DATA_PATH}'
           f' --save_path {SAVE_PATH}'
           f' --epochs {EPOCHS}'
           f' --batch_size {BATCH_SIZE}'
           f' --learning_rate {LR}')
print('Command:', cmd_res)

In [ ]:
# ── 執行 ResNet34-UNet 訓練 ───────────────────────────────────────────
os.makedirs(SAVE_PATH, exist_ok=True)
!{cmd_res}

## 9. 推論 — 競賽一：UNet

In [ ]:
# ── 9. UNet 推論 ──────────────────────────────────────────────────────
cmd_inf_unet = (
    'python src/inference.py'
    ' --model unet'
    ' --data_path dataset/oxford-iiit-pet'
    ' --weight /content/drive/MyDrive/lab2/saved_models/unet_best.pth'
    ' --output submission_unet.csv'
    ' --batch_size 16'
)
print('Command:', cmd_inf_unet)
!{cmd_inf_unet}

In [ ]:
# ── 下載 UNet submission ──────────────────────────────────────────────
import pandas as pd, shutil
from google.colab import files

df = pd.read_csv('submission_unet.csv')
print(f'Shape: {df.shape}')
print(df.head())
shutil.copy('submission_unet.csv', '/content/drive/MyDrive/lab2/submission_unet.csv')
files.download('submission_unet.csv')

## 10. 推論 — 競賽二：ResNet34-UNet

In [ ]:
# ── 10. ResNet34-UNet 推論 ────────────────────────────────────────────
cmd_inf_res = (
    'python src/inference.py'
    ' --model resnet34_unet'
    ' --data_path dataset/oxford-iiit-pet'
    ' --weight /content/drive/MyDrive/lab2/saved_models/resnet34_unet_best.pth'
    ' --output submission_resnet34_unet.csv'
    ' --batch_size 16'
)
print('Command:', cmd_inf_res)
!{cmd_inf_res}

In [ ]:
# ── 下載 ResNet34-UNet submission ─────────────────────────────────────
import pandas as pd, shutil
from google.colab import files

df = pd.read_csv('submission_resnet34_unet.csv')
print(f'Shape: {df.shape}')
print(df.head())
shutil.copy('submission_resnet34_unet.csv', '/content/drive/MyDrive/lab2/submission_resnet34_unet.csv')
files.download('submission_resnet34_unet.csv')